# CIFAR-10N: T4 GCE + T5 CE baseline (synthetic)
Runtime → Change runtime type → **T4 GPU**. Add a Colab secret (🔑 sidebar) named `KAGGLE_API_TOKEN`
with your Kaggle API token (same value as `~/.kaggle/access_token`), notebook access ON. Then Run all.

In [ ]:
import os
from google.colab import userdata, drive
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
drive.mount("/content/drive")
!nvidia-smi -L
!pip -q install -U kaggle
!kaggle datasets download jainamrampuria/cifar10n -p /content/data --unzip -q
!kaggle datasets download jainamrampuria/cv-code  -p /content/cv   --unzip -q
!ls /content/data /content/cv

In [ ]:
%%bash
# same setup as every other run; only the method / label column changes. Both runs share the one T4.
cd /content/cv
python train.py --data ../data --out ../outputs/t4_gce          --label_col noisy_label --loss gce > ../t4_gce.log 2>&1 &
python train.py --data ../data --out ../outputs/t5_baseline_synth --label_col synthetic_label --loss ce > ../t5_base.log 2>&1 &
wait
tail -n 2 ../t4_gce.log ../t5_base.log

In [ ]:
%%bash
cd /content/cv
for r in t4_gce t5_baseline_synth; do
  python make_preds.py --run ../outputs/$r --epoch final --out ../predictions/${r}_final.csv
  python make_preds.py --run ../outputs/$r --epoch best  --out ../predictions/${r}_best.csv
done
cat ../outputs/t4_gce/summary.json ../outputs/t5_baseline_synth/summary.json
D=/content/drive/MyDrive/cifar10n_runs; mkdir -p $D
cp -r ../outputs/t4_gce ../outputs/t5_baseline_synth ../predictions ../t4_gce.log ../t5_base.log $D/
ls -R $D | head -40